# Tugas 2

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    precision_score, recall_score, f1_score
)

In [2]:
df = pd.read_csv('spam.csv', encoding='latin-1')
df = df.drop(df.iloc[:, 2:], axis=1)
df = df.rename(columns={'v1': 'Labels', 'v2': 'SMS'})
df['Labels'] = df['Labels'].map({'ham': 0, 'spam': 1})
df.head()

,Labels,SMS
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."


In [3]:
X = df['SMS'].to_numpy()
y = df['Labels'].to_numpy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=50
)
print('Jumlah data latih:', len(X_train))
print('Jumlah data uji:', len(X_test))

Jumlah data latih: 4457
Jumlah data uji: 1115


In [4]:
# Multinomial Naive Bayes dengan CountVectorizer dan stop words
bow = CountVectorizer(stop_words='english')
X_train_count = bow.fit_transform(X_train)
X_test_count = bow.transform(X_test)

mnb_count = MultinomialNB()
mnb_count.fit(X_train_count, y_train)
y_train_pred_count = mnb_count.predict(X_train_count)
y_test_pred_count = mnb_count.predict(X_test_count)

print('Jumlah fitur CountVectorizer:', len(bow.get_feature_names_out()))

Jumlah fitur CountVectorizer: 7466


In [5]:
acc_train_count = accuracy_score(y_train, y_train_pred_count)
acc_test_count = accuracy_score(y_test, y_test_pred_count)

print('CountVectorizer dengan stop_words=english')
print('Akurasi data train:', acc_train_count)
print('Akurasi data test:', acc_test_count)
print('Confusion Matrix (urutan: ham, spam):')
print(confusion_matrix(y_test, y_test_pred_count, labels=[0, 1]))
print('Laporan Klasifikasi:')
print(classification_report(y_test, y_test_pred_count, labels=[0, 1], target_names=['ham', 'spam']))

CountVectorizer dengan stop_words=english
Akurasi data train: 0.9946152120260264
Akurasi data test: 0.9829596412556054
Confusion Matrix (urutan: ham, spam):
[[951   3]
 [ 16 145]]
Laporan Klasifikasi:
              precision    recall  f1-score   support

         ham       0.98      1.00      0.99       954
        spam       0.98      0.90      0.94       161

    accuracy                           0.98      1115
   macro avg       0.98      0.95      0.96      1115
weighted avg       0.98      0.98      0.98      1115



In [6]:
# Multinomial Naive Bayes dengan TF-IDF dan stop words
# Gunakan pembagian data yang sama dengan model CountVectorizer.
tfidf = TfidfVectorizer(stop_words='english')
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

mnb_tfidf = MultinomialNB()
mnb_tfidf.fit(X_train_tfidf, y_train)
y_train_pred_tfidf = mnb_tfidf.predict(X_train_tfidf)
y_test_pred_tfidf = mnb_tfidf.predict(X_test_tfidf)

print('Jumlah fitur TF-IDF:', len(tfidf.get_feature_names_out()))

Jumlah fitur TF-IDF: 7466


In [7]:
acc_train_tfidf = accuracy_score(y_train, y_train_pred_tfidf)
acc_test_tfidf = accuracy_score(y_test, y_test_pred_tfidf)

print('TF-IDF dengan stop_words=english')
print('Akurasi data train:', acc_train_tfidf)
print('Akurasi data test:', acc_test_tfidf)
print('Confusion Matrix (urutan: ham, spam):')
print(confusion_matrix(y_test, y_test_pred_tfidf, labels=[0, 1]))
print('Laporan Klasifikasi:')
print(classification_report(y_test, y_test_pred_tfidf, labels=[0, 1], target_names=['ham', 'spam']))

TF-IDF dengan stop_words=english
Akurasi data train: 0.9842943684092439
Akurasi data test: 0.9605381165919282
Confusion Matrix (urutan: ham, spam):
[[954   0]
 [ 44 117]]
Laporan Klasifikasi:


              precision    recall  f1-score   support

         ham       0.96      1.00      0.98       954
        spam       1.00      0.73      0.84       161

    accuracy                           0.96      1115
   macro avg       0.98      0.86      0.91      1115
weighted avg       0.96      0.96      0.96      1115



In [8]:
hasil_evaluasi = pd.DataFrame({
    'Akurasi train': [acc_train_count, acc_train_tfidf],
    'Akurasi test': [acc_test_count, acc_test_tfidf],
    'Precision spam': [
        precision_score(y_test, y_test_pred_count),
        precision_score(y_test, y_test_pred_tfidf)
    ],
    'Recall spam': [
        recall_score(y_test, y_test_pred_count),
        recall_score(y_test, y_test_pred_tfidf)
    ],
    'F1-score spam': [
        f1_score(y_test, y_test_pred_count),
        f1_score(y_test, y_test_pred_tfidf)
    ]
}, index=['CountVectorizer + stop_words', 'TF-IDF + stop_words'])
hasil_evaluasi

,Akurasi train,Akurasi test,Precision spam,Recall spam,F1-score spam
CountVectorizer + stop_words,0.994615,0.982960,0.97973,0.900621,0.938511
TF-IDF + stop_words,0.984294,0.960538,1.00000,0.726708,0.841727


Kedua model menggunakan Multinomial Naive Bayes dengan parameter bawaan, stop words bahasa Inggris, dan pembagian data yang sama: 80% data latih serta 20% data uji dengan random_state=50. Vectorizer hanya dilatih pada data latih.

CountVectorizer menghasilkan akurasi test 98,30%, precision spam 97,97%, recall spam 90,06%, dan F1-score spam 93,85%. TF-IDF menghasilkan akurasi test 96,05%, precision spam 100,00%, recall spam 72,67%, dan F1-score spam 84,17%. Pada confusion matrix, CountVectorizer melewatkan 16 SMS spam sebagai ham, sedangkan TF-IDF melewatkan 44 SMS spam.

Pada percobaan ini, fitur terbaik berdasarkan F1-score kelas spam adalah CountVectorizer + stop_words. F1-score dipakai sebagai dasar kesimpulan karena mempertimbangkan precision dan recall pada kelas spam yang jumlahnya lebih sedikit daripada ham. Kesimpulan ini berlaku untuk pembagian data dan parameter yang diuji.